# Legacy exploratory notebook — superseded
This notebook predates the audited engine and is retained only as development history. Its outputs are not current research results. In particular, early backtest notebooks contain obsolete timing, missing-return filtering, and drawdown calculations. Do not use them to validate performance or overwrite the bundled data. Use the root Python entrypoints and [methodology](../docs/METHODOLOGY.md).

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

backtest = pd.read_parquet(
    "../data/processed/backtest.parquet"
)

backtest.head()

benchmark = pd.read_parquet(

    "../data/raw/benchmark.parquet"

)

benchmark_close = benchmark["Close"].copy()

benchmark_close.head()

Ticker,^GSPC
Date,
2015-01-02,2058.199951
2015-01-05,2020.579956
2015-01-06,2002.609985
2015-01-07,2025.900024
2015-01-08,2062.139893


In [12]:
backtest[["date", "holdings", "portfolio_return"]].head(12)

,date,holdings,portfolio_return
0,2015-04-30,"[MSFT, AMZN, JPM, NVDA, TSLA]",0.027623
1,2015-05-31,"[TSLA, AMZN, ABBV, JPM, MSFT]",0.012444
2,2015-06-30,"[TSLA, AMZN, ABBV, JPM, MSFT]",0.070536
3,2015-07-31,"[AMZN, GOOGL, META, TSLA, V]",-0.044622
4,2015-08-31,"[AMZN, GOOGL, META, HD, V]",-0.007512
5,2015-09-30,"[NVDA, GOOGL, AMZN, COST, META]",0.151347
6,2015-10-31,"[NVDA, AMZN, MSFT, GOOGL, COST]",0.056392
7,2015-11-30,"[NVDA, AMZN, MSFT, GOOGL, META]",0.020093
8,2015-12-31,"[NVDA, AMZN, MSFT, GOOGL, META]",-0.039834
9,2016-01-31,"[WMT, META, PG, MSFT, NVDA]",-0.011960


In [13]:
benchmark_monthly = (
    benchmark_close
    .resample("ME")
    .last()
    .pct_change(fill_method=None)
)

if isinstance(benchmark_monthly, pd.DataFrame):
    benchmark_monthly = benchmark_monthly.iloc[:, 0]

benchmark_monthly = benchmark_monthly.reindex(backtest["date"]).dropna()

benchmark_annual_return = (
    (1 + benchmark_monthly).prod()
    ** (12 / len(benchmark_monthly))
    - 1
)

benchmark_volatility = benchmark_monthly.std() * np.sqrt(12)

benchmark_sharpe = (
    benchmark_monthly.mean()
    / benchmark_monthly.std()
) * np.sqrt(12)

benchmark_wealth = (1 + benchmark_monthly).cumprod()
benchmark_drawdown = benchmark_wealth / benchmark_wealth.cummax() - 1
benchmark_max_drawdown = benchmark_drawdown.min()

In [14]:
strategy_returns = backtest["portfolio_return"].dropna()

annual_return = (
    (1 + strategy_returns).prod()
    ** (12 / len(strategy_returns))
    - 1
)

annual_volatility = (
    strategy_returns.std()
    * np.sqrt(12)
)

sharpe_ratio = (
    strategy_returns.mean()
    / strategy_returns.std()
) * np.sqrt(12)

strategy_wealth = (
    1 + strategy_returns
).cumprod()

strategy_drawdown = (
    strategy_wealth
    / strategy_wealth.cummax()
    - 1
)

max_drawdown = strategy_drawdown.min()

In [15]:
comparison = pd.DataFrame({
    "Momentum Strategy": [
        annual_return,
        annual_volatility,
        sharpe_ratio,
        max_drawdown
    ],
    "S&P 500": [
        benchmark_annual_return,
        benchmark_volatility,
        benchmark_sharpe,
        benchmark_max_drawdown
    ]
}, index=[
    "Annual Return",
    "Annual Volatility",
    "Sharpe Ratio",
    "Max Drawdown"
])

comparison

,Momentum Strategy,S&P 500
Annual Return,0.345331,0.118819
Annual Volatility,0.216053,0.150585
Sharpe Ratio,1.492324,0.824132
Max Drawdown,-0.203464,-0.247695
